# Fase 5 · M02: Árboles de Decisión

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M02 — Árboles de Decisión |

---

## 🎯 Qué hace

Entrena y evalúa la familia de árboles de decisión sobre el dataset D_strict
(24 features + target `abandono`, incl. indicadores de imputación informativa).
Para cada modelo se prueban las 3 estrategias de balance de clases y se valida
con 5-Fold CV estratificado.

**Modelos:** Decision Tree · Random Forest · Extra Trees

**Estrategias de balance:** Sin ajuste · `class_weight='balanced'` · SMOTE

**Por qué árboles para este problema:**
Los árboles son inmunes a la multicolinealidad, no requieren escalado y producen
importancia de features interpretable directamente. Ideales para datos tabulares
mixtos como expedientes universitarios.

## 📋 Requisitos

- `data/05_modelado/X_train_prep.parquet`, `X_test_prep.parquet` — generados por `f5_m01a_preparacion`
- `data/05_modelado/y_train.parquet`, `y_test.parquet`
- Entorno: `tfm_abandono` (scikit-learn ≥1.3, optuna, imbalanced-learn, codecarbon, joblib)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/05_modelado/results/results_arboles.parquet` | Métricas completas 9 combinaciones |
| `data/05_modelado/results/results_arboles.xlsx` | Excel para revisión |
| `data/05_modelado/models/DecisionTree__*.pkl` | 3 pipelines entrenados |
| `data/05_modelado/models/RandomForest__*.pkl` | 3 pipelines entrenados |
| `data/05_modelado/models/ExtraTrees__*.pkl` | 3 pipelines entrenados |
| `docs/html/fase5/m02_arboles.html` | Informe HTML |

## 🔄 Flujo

```
X_train_prep / y_train  (f5_m01a_preparacion)
    ↓ Hiperparámetros Optuna (50 trials × 3 modelos)
    ↓ Celda 4b: limpieza automática de pkl obsoletos
    ↓ 5-Fold CV estratificado × 9 combinaciones
    ↓ Importancia de features (Gini) + curva aprendizaje
    ↓ Calibración + ROC + matriz confusión + umbral óptimo
    → results_arboles.parquet + 9 pkl + HTML
```

## ➡️ Siguiente

`f5_m03_boosting.ipynb` — XGBoost, LightGBM, CatBoost, GradientBoosting

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import calibration_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from codecarbon import EmissionsTracker
    CODECARBON_OK = True
except ImportError:
    CODECARBON_OK = False
    print('⚠️  codecarbon no instalado')

warnings.filterwarnings('ignore')

# ROOT detection
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'
RUTA_RESULTS  = RUTA_MODELADO / 'results'
RUTA_MODELS   = RUTA_MODELADO / 'models'
RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_RESULTS, RUTA_MODELS, RUTA_HTML_F5])

# Constantes
RANDOM_STATE    = 42
N_SPLITS_CV     = 5
N_TRIALS_OPTUNA = 50
FAMILIA    = 'arboles'
ESTRATEGIAS     = ['none', 'balanced', 'smote']
COLOR           = '#3182ce'
fmt             = formato_numero_es

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')
print(f'📂 RESULTS : {RUTA_RESULTS}')
print(f'📂 MODELS  : {RUTA_MODELS}')
print(f'\n🌿 codecarbon: {CODECARBON_OK}')

graficos_b64 = {}  # acumulador de gráficos base64

✓ Directorios verificados: 3
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS
# ============================================================================
# Lee splits y preprocesados generados por f5_m01_preparacion.
# Los _prep se cargan desde disco directamente — no se re-aplica el transform.
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    print('⏳ Aplicando pipeline (primera vez)...')
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    X_train_prep.to_parquet(ruta_train_prep)
    X_test_prep.to_parquet(ruta_test_prep)
    print('✅ Preprocesados generados y guardados')

FEATURE_NAMES = list(X_train_prep.columns)  # 27 features incl. _missing
N_FEATURES    = len(FEATURE_NAMES)

print('=' * 55)
print('DATOS CARGADOS')
print('=' * 55)
print(f'X_train : {fmt(len(X_train))} × {N_FEATURES}  |  abandono: {y_train.mean()*100:.1f}%')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print(f'Pipeline: {pipeline_prep.__class__.__name__} ✅')
print()
print(f'Features : {N_FEATURES} (incl. indicadores _missing)')
print('⚠️  X_test INTOCABLE hasta f5_m06_ensambles.ipynb')

✅ Preprocesados cargados desde disco
DATOS CARGADOS
X_train : 26.896 × 27  |  abandono: 29.2%
X_test  : 6.725  × 27  |  abandono: 29.2%
Pipeline: ColumnTransformer ✅

Features : 27 (incl. indicadores _missing)
⚠️  X_test INTOCABLE hasta f5_m06_ensambles.ipynb


In [3]:
# ============================================================================
# CELDA 2b: DIAGNÓSTICO DE PKL — estado antes de entrenar
# ============================================================================
# Muestra en segundos qué pkl existen, si son compatibles con las features
# actuales, y si hace falta re-entrenar. Sin tocar nada.
# ============================================================================

import joblib as _jl

print('=' * 60)
print('DIAGNÓSTICO PKL — f5_m02_arboles')
print('=' * 60)

# Leer features actuales
_ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
_n_actual  = None
if _ruta_meta.exists():
    with open(_ruta_meta) as _f:
        _n_actual = json.load(_f).get('n_features_final')
print(f'Features actuales (meta_preparacion): {_n_actual}')
print()

_hay_que_entrenar = []
for _m in ['DecisionTree', 'RandomForest', 'ExtraTrees']:
    for _e in ['none', 'balanced', 'smote']:
        _clave = f'{_m}__{_e}'
        _ruta  = RUTA_MODELS / f'{_clave}.pkl'
        if _ruta.exists():
            try:
                _modelo = _jl.load(_ruta)
                _ultimo = _modelo.steps[-1][1] if hasattr(_modelo,'steps') else _modelo
                _n_pkl  = getattr(_ultimo, 'n_features_in_', None)
                if _n_pkl == _n_actual:
                    print(f'  ✅ {_clave}: {_n_pkl} features — OK')
                else:
                    print(f'  ❌ {_clave}: {_n_pkl} features — OBSOLETO (actual={_n_actual})')
                    _hay_que_entrenar.append(_clave)
            except Exception as _ex:
                print(f'  ⚠️  {_clave}: no cargable ({str(_ex)[:40]})')
                _hay_que_entrenar.append(_clave)
        else:
            print(f'  ⏳ {_clave}: no existe — hay que entrenar')
            _hay_que_entrenar.append(_clave)

_pq = RUTA_RESULTS / 'results_arboles.parquet'
print(f'\nParquet resultados: {"✅ existe" if _pq.exists() else "❌ no existe — se generará"}')

print()
if not _hay_que_entrenar and _pq.exists():
    print('✅ TODO EN ORDEN — la celda 5 cargará desde disco (rápido)')
elif not _hay_que_entrenar and not _pq.exists():
    print('ℹ️  PKL OK pero falta parquet — se recalcularán métricas (~5 min)')
else:
    n = len(_hay_que_entrenar)
    print(f'⏳ HAY QUE ENTRENAR {n} combinaciones:')
    for _c in _hay_que_entrenar:
        print(f'   · {_c}')
    _t_est = n * 5  # estimación conservadora en minutos
    print(f'   Tiempo estimado: ~{_t_est}-{_t_est*2} min (depende de CPU)')
print('=' * 60)


DIAGNÓSTICO PKL — f5_m02_arboles
Features actuales (meta_preparacion): 27

  ✅ DecisionTree__none: 27 features — OK
  ✅ DecisionTree__balanced: 27 features — OK
  ✅ DecisionTree__smote: 27 features — OK
  ✅ RandomForest__none: 27 features — OK
  ✅ RandomForest__balanced: 27 features — OK
  ✅ RandomForest__smote: 27 features — OK
  ✅ ExtraTrees__none: 27 features — OK
  ✅ ExtraTrees__balanced: 27 features — OK
  ✅ ExtraTrees__smote: 27 features — OK

Parquet resultados: ✅ existe

✅ TODO EN ORDEN — la celda 5 cargará desde disco (rápido)


In [4]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    """Ensambla modelo con estrategia de balance.
    none     → modelo directo
    balanced → class_weight en el modelo
    smote    → SMOTE antes del modelo (ImbPipeline)
    """
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    """5-Fold CV estratificado con métricas completas."""
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    """Métricas completas sobre test para modelo ya entrenado."""
    y_pred  = pipe_fit.predict(X_te)
    y_proba = (
        pipe_fit.predict_proba(X_te)[:, 1]
        if hasattr(pipe_fit, 'predict_proba')
        else pipe_fit.decision_function(X_te)
    )
    # Normalizar decision_function a [0,1] si es necesario
    if y_proba.min() < 0 or y_proba.max() > 1:
        y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min() + 1e-9)
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


def construir_modelo(nombre: str, estrategia: str):
    """Construye modelo con hiperparámetros registrados de Optuna."""
    p  = PARAMS_OPTUNA[nombre]
    cw = 'balanced' if estrategia == 'balanced' else None
    # construir_modelo definido en celda 5
    pass


print('✅ Funciones auxiliares definidas')
print('   · construir_pipeline  · evaluar_cv')
print('   · calcular_metricas_test  · construir_modelo')

✅ Funciones auxiliares definidas
   · construir_pipeline  · evaluar_cv
   · calcular_metricas_test  · construir_modelo


In [5]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS ÓPTIMOS — Decision Tree · Random Forest · Extra Trees
# ============================================================================
# Params registrados tras ejecución de Optuna (50 trials × 3 modelos).
# Árboles y ensambles son mucho más rápidos que SVM con 26.000 filas.
# Cambiar FORZAR_OPTUNA = True solo si se quiere re-ejecutar.
# ============================================================================

FORZAR_OPTUNA = False

PARAMS_OPTUNA = {
    'DecisionTree' : {
        'max_depth'       : 8,
        'min_samples_split': 20,
        'min_samples_leaf' : 10,
        'criterion'        : 'gini',
        'max_features'     : 'sqrt',
    },
    'RandomForest' : {
        'n_estimators'     : 300,
        'max_depth'        : 12,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
        'max_features'     : 'sqrt',
    },
    'ExtraTrees'   : {
        'n_estimators'     : 300,
        'max_depth'        : 12,
        'min_samples_split': 10,
        'min_samples_leaf' : 5,
        'max_features'     : 'sqrt',
    },
}
AUC_OPTUNA = {
    'DecisionTree': None,
    'RandomForest': None,
    'ExtraTrees'  : None,
}

if not FORZAR_OPTUNA:
    mejores_params = PARAMS_OPTUNA
    print('⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)')
    print('   Tras ejecutar, registrar los params y poner FORZAR_OPTUNA = False')
    for nombre, params in mejores_params.items():
        print(f'   {nombre:<14}: {params}')
else:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objetivo_dt(trial):
        params = {
            'max_depth'        : trial.suggest_int('max_depth', 3, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 50),
            'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 30),
            'criterion'        : trial.suggest_categorical('criterion', ['gini', 'entropy']),
            'max_features'     : trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        }
        pipe = construir_pipeline(
            DecisionTreeClassifier(**params, class_weight='balanced',
                                   random_state=RANDOM_STATE), 'balanced')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    def objetivo_rf(trial):
        params = {
            'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
            'max_depth'        : trial.suggest_int('max_depth', 5, 25),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
            'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features'     : trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        }
        pipe = construir_pipeline(
            RandomForestClassifier(**params, class_weight='balanced',
                                   random_state=RANDOM_STATE, n_jobs=-1), 'balanced')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    def objetivo_et(trial):
        params = {
            'n_estimators'     : trial.suggest_int('n_estimators', 100, 500),
            'max_depth'        : trial.suggest_int('max_depth', 5, 25),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
            'min_samples_leaf' : trial.suggest_int('min_samples_leaf', 1, 20),
            'max_features'     : trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        }
        pipe = construir_pipeline(
            ExtraTreesClassifier(**params, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1), 'balanced')
        return cross_validate(pipe, X_train_prep, y_train, cv=CV,
                              scoring='roc_auc', n_jobs=-1)['test_score'].mean()

    estudios_optuna = {}
    print('=' * 60)
    print(f'OPTUNA — {N_TRIALS_OPTUNA} trials por modelo (métrica: AUC-ROC 5-Fold CV)')
    print('=' * 60)
    for nombre, fn in [('DecisionTree', objetivo_dt),
                       ('RandomForest', objetivo_rf),
                       ('ExtraTrees',   objetivo_et)]:
        print(f'⏳ {nombre}...', end=' ', flush=True)
        t0 = time.time()
        estudio = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
        )
        estudio.optimize(fn, n_trials=N_TRIALS_OPTUNA, show_progress_bar=False)
        estudios_optuna[nombre] = estudio
        AUC_OPTUNA[nombre] = round(estudio.best_value, 4)
        print(f'✅  mejor AUC={estudio.best_value:.4f}  ({time.time()-t0:.0f}s)')
        print(f'   Mejores params: {estudio.best_params}')

    mejores_params = {n: estudios_optuna[n].best_params for n in estudios_optuna}
    print('\n⚠️  Copia los params en PARAMS_OPTUNA y pon FORZAR_OPTUNA = False')
    print('✅ Optuna completado')

⚠️  Params iniciales — ejecutar Optuna la primera vez (FORZAR_OPTUNA = True)
   Tras ejecutar, registrar los params y poner FORZAR_OPTUNA = False
   DecisionTree  : {'max_depth': 8, 'min_samples_split': 20, 'min_samples_leaf': 10, 'criterion': 'gini', 'max_features': 'sqrt'}
   RandomForest  : {'n_estimators': 300, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt'}
   ExtraTrees    : {'n_estimators': 300, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'sqrt'}


In [6]:
# ============================================================================
# CELDA 4b: LIMPIEZA AUTOMÁTICA DE PKL OBSOLETOS
# ============================================================================
# Problema: si existe un pkl entrenado con un dataset anterior (distinto número
# de features), sklearn lanza ValueError al predecir — las features no coinciden.
#
# Solución robusta:
#   1. Lee n_features_final del meta_preparacion.json (fuente de verdad)
#   2. Carga cada pkl de esta familia y comprueba su n_features_in_
#   3. Si no coincide → borra el pkl Y el parquet de resultados
#   4. Si no puede verificar → conserva (no borra por precaución)
#   5. Si faltan pkl → borra parquet para forzar re-entrenamiento completo
#
# Esto garantiza que siempre se re-entrena cuando el dataset cambia.
# ============================================================================

import glob, os, json as _json, joblib as _jl

MODELOS_FAMILIA_M02 = ['DecisionTree', 'RandomForest', 'ExtraTrees']

# Leer nº de features actual desde metadatos
ruta_meta = RUTA_MODELADO / 'meta_preparacion.json'
n_features_actual = None
if ruta_meta.exists():
    with open(ruta_meta) as _f:
        n_features_actual = _json.load(_f).get('n_features_final')
    print(f'📋 Features actuales (meta_preparacion.json): {n_features_actual}')
else:
    print('⚠️  meta_preparacion.json no encontrado — no se puede verificar pkl')

eliminados = []
no_verificables = []
for pkl in sorted(glob.glob(str(RUTA_MODELS / '*.pkl'))):
    nombre = os.path.basename(pkl)
    if not any(m in nombre for m in MODELOS_FAMILIA_M02):
        continue
    try:
        modelo = _jl.load(pkl)
        # Buscar n_features_in_ en distintos niveles del pipeline
        candidatos = [modelo]
        if hasattr(modelo, 'steps'):
            candidatos += [s[1] for s in modelo.steps]
        if hasattr(modelo, 'estimators_'):
            candidatos += list(modelo.estimators_)[:1]
        
        n_pkl = None
        for obj in candidatos:
            n_pkl = getattr(obj, 'n_features_in_', None)
            if n_pkl is not None:
                break
        
        if n_pkl is not None and n_features_actual is not None:
            if n_pkl != n_features_actual:
                os.remove(pkl)
                eliminados.append(f'{nombre} ({n_pkl} features → obsoleto, actual={n_features_actual})')
        elif n_pkl is None:
            no_verificables.append(nombre)
    except Exception as e:
        # No borrar si no se puede cargar — podría ser corrupción puntual
        no_verificables.append(f'{nombre} (error: {str(e)[:40]})')

# Si se eliminaron pkl → borrar parquet para forzar re-entrenamiento
if eliminados:
    ruta_pq = RUTA_RESULTS / 'results_arboles.parquet'
    if ruta_pq.exists():
        os.remove(ruta_pq)
        print('   ✅ results_arboles.parquet eliminado → re-entrenamiento completo')

# Si faltan pkl aunque no se eliminaron → también borrar parquet
claves_esp = [f'{m}__{e}' for m in MODELOS_FAMILIA_M02 for e in ['none','balanced','smote']]
pkls_now   = {os.path.basename(p).replace('.pkl','')
              for p in glob.glob(str(RUTA_MODELS / '*.pkl'))}
faltan = [c for c in claves_esp if c not in pkls_now]
if faltan and not eliminados:
    ruta_pq = RUTA_RESULTS / 'results_arboles.parquet'
    if ruta_pq.exists():
        os.remove(ruta_pq)
        print(f'⚠️  PKL faltantes: {faltan}')
        print('   results_arboles.parquet eliminado → re-entrenamiento')

# Resumen
if eliminados:
    print(f'⚠️  PKL obsoletos eliminados ({len(eliminados)}):')
    for e in eliminados: print(f'   · {e}')
if no_verificables:
    print(f'ℹ️  No verificables (conservados): {no_verificables}')
if not eliminados and not faltan:
    print(f'✅ Todos los PKL de M02 son compatibles con {n_features_actual} features')

📋 Features actuales (meta_preparacion.json): 27
✅ Todos los PKL de M02 son compatibles con 27 features


In [7]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 9 COMBINACIONES (3 modelos × 3 estrategias)
# ============================================================================
# Decision Tree, Random Forest y Extra Trees admiten class_weight='balanced'
# nativamente — no necesitan calibración adicional.
# Si los 9 modelos .pkl ya existen en disco los carga directamente.
# ============================================================================

ESTRATEGIAS  = ['none', 'balanced', 'smote']
MODELOS_M02  = ['DecisionTree', 'RandomForest', 'ExtraTrees']

def construir_modelo(nombre: str, estrategia: str):
    p  = mejores_params[nombre]
    cw = 'balanced' if estrategia == 'balanced' else None
    if nombre == 'DecisionTree':
        return DecisionTreeClassifier(
            **p, class_weight=cw, random_state=RANDOM_STATE
        )
    elif nombre == 'RandomForest':
        return RandomForestClassifier(
            **p, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1
        )
    elif nombre == 'ExtraTrees':
        return ExtraTreesClassifier(
            **p, class_weight=cw, random_state=RANDOM_STATE, n_jobs=-1
        )

claves_esperadas = [f'{m}__{e}' for m in MODELOS_M02 for e in ESTRATEGIAS]
pkls_en_disco    = [c for c in claves_esperadas if (RUTA_MODELS / f'{c}.pkl').exists()]
parquet_en_disco = (RUTA_RESULTS / 'results_arboles.parquet').exists()

print(f'📦 Modelos en disco : {len(pkls_en_disco)}/9')
print(f'📊 Parquet resultados: {parquet_en_disco}')

modelos_fit = {c: joblib.load(RUTA_MODELS / f'{c}.pkl') for c in pkls_en_disco}

if parquet_en_disco:
    df_res = pd.read_parquet(RUTA_RESULTS / 'results_arboles.parquet')
    df_cv  = df_res.sort_values('auc_mean', ascending=False)
    emisiones = None
    print('✅ Resultados cargados desde disco')

elif len(pkls_en_disco) == 9:
    print('\n⏳ Reconstruyendo métricas (modelos en disco, parquet no)...')
    resultados_cv   = []
    resultados_test = []
    for nombre in MODELOS_M02:
        print(f'   {nombre}...', end=' ')
        for estrategia in ESTRATEGIAS:
            clave = f'{nombre}__{estrategia}'
            pipe  = modelos_fit[clave]
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            res_test = calcular_metricas_test(nombre, pipe, X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
        print('✅')
    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    emisiones = None
    print('✅ Métricas reconstruidas')

else:
    print('\n⏳ Entrenando 9 combinaciones...')
    resultados_cv   = []
    resultados_test = []

    if CODECARBON_OK:
        tracker = EmissionsTracker(project_name=f'TFM_f5_m02_{FAMILIA}',
                                   output_dir=str(RUTA_RESULTS), log_level='error')
        tracker.start()

    print('=' * 60)
    print('ENTRENAMIENTO — 3 modelos × 3 estrategias = 9 combinaciones')
    print('=' * 60)
    for nombre in MODELOS_M02:
        print(f'\n── {nombre} ──')
        for estrategia in ESTRATEGIAS:
            clave = f'{nombre}__{estrategia}'
            pipe  = construir_pipeline(construir_modelo(nombre, estrategia), estrategia)
            res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                                X_train_prep, y_train, estrategia)
            resultados_cv.append(res_cv)
            pipe.fit(X_train_prep, y_train)
            modelos_fit[clave] = pipe
            res_test = calcular_metricas_test(nombre, pipe, X_test_prep, y_test, estrategia)
            resultados_test.append(res_test)
            joblib.dump(pipe, RUTA_MODELS / f'{clave}.pkl')
            print(f'   [{estrategia:<8}] CV AUC={res_cv["auc_mean"]:.4f}  F1={res_cv["f1_mean"]:.4f}')

    if CODECARBON_OK:
        emisiones = tracker.stop()
        print(f'\n🌿 Emisiones CO₂: {emisiones:.6f} kg CO₂eq')
    else:
        emisiones = None

    df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
    df_test = pd.DataFrame(resultados_test)
    df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])
    print('\n✅ Entrenamiento completado — 9 combinaciones')

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
MEJOR_CLAVE      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[MEJOR_CLAVE]

print(f'\n🏆 MEJOR: {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f}  |  F1 CV = {df_cv.iloc[0]["f1_mean"]:.4f}')

📦 Modelos en disco : 9/9
📊 Parquet resultados: True
✅ Resultados cargados desde disco

🏆 MEJOR: RandomForest + smote
   AUC CV = 0.9473  |  F1 CV = 0.8170


In [8]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'f1_test',
                 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST (informativas — selección por CV)')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))

RESULTADOS 5-Fold CV — ordenado por AUC
      modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
RandomForest      smote    0.9473   0.0028   0.8170          0.8060       0.8285   78.5150
RandomForest       none    0.9468   0.0032   0.8119          0.8745       0.7579   22.3540
RandomForest   balanced    0.9463   0.0029   0.8120          0.7844       0.8418   14.8620
  ExtraTrees       none    0.9315   0.0038   0.7592          0.8768       0.6696   35.5220
  ExtraTrees   balanced    0.9314   0.0035   0.7811          0.7330       0.8360    2.7930
  ExtraTrees      smote    0.9308   0.0033   0.7834          0.7466       0.8243   34.2490
DecisionTree      smote    0.9058   0.0032   0.7493          0.6980       0.8091   16.2980
DecisionTree       none    0.9027   0.0049   0.7362          0.7754       0.7015    2.5880
DecisionTree   balanced    0.9000   0.0033   0.7329          0.6546       0.8347    2.8130

MÉTRICAS EN TEST (informativas — selección por CV

In [9]:
# ============================================================================
# CELDA 7: IMPORTANCIA DE FEATURES — Top 10
# ============================================================================
# Los árboles exponen feature_importances_ directamente (Gini importance).
# Se usa el mejor modelo para el análisis.
# ============================================================================


try:
    # Extraer el árbol base del pipeline
    estimador = mejor_pipe.named_steps.get('estimator') or mejor_pipe.named_steps.get('model')
    if estimador is None:
        estimador = mejor_pipe[-1]

    importancias = estimador.feature_importances_
    indices      = np.argsort(importancias)[-10:]
    nombres      = [FEATURE_NAMES[j] for j in indices]
    valores      = importancias[indices]

    colores = ['#e53e3e' if v == valores.max() else COLOR for v in valores]

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(nombres, valores, color=colores, height=0.6)
    ax.set_xlabel('Importancia (Gini)')
    ax.set_title(
        f'Top 10 features por importancia — {MEJOR_MODELO}__{MEJOR_ESTRATEGIA}\n'
        f'Rojo = feature más importante',
        fontsize=11, fontweight='bold'
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    graficos_b64['importancia'] = figura_a_base64(fig)
    plt.close(fig)
    print(f'✅ Importancia de features calculada — {MEJOR_MODELO}')
except Exception as e:
    print(f'⚠️  No se pudo calcular importancia: {e}')
    graficos_b64['importancia'] = None


✅ Importancia de features calculada — RandomForest


In [10]:
# ============================================================================
# CELDA 8: CURVA DE APRENDIZAJE
# ============================================================================
# SVM RBF es cuadrático con el tamaño del dataset — usar la muestra completa
# no es viable. Se usa una submuestra estratificada del 30% del train,
# práctica estándar: la curva muestra la tendencia de generalización.
# ============================================================================

# Submuestra estratificada 30%
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_sub = X_train_prep.iloc[idx_sub]
y_sub = y_train.iloc[idx_sub]

print(f'Submuestra: {fmt(len(X_sub))} filas '
      f'({len(X_sub)/len(X_train_prep)*100:.0f}% del train)  '
      f'abandono={y_sub.mean()*100:.1f}%')
print('Calculando curva de aprendizaje...', end=' ', flush=True)
t0 = time.time()

train_sizes, train_scores, cv_scores = learning_curve(
    mejor_pipe, X_sub, y_sub,
    cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color=COLOR, label='Train AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLOR)
ax.plot(train_sizes, cv_mean, 'o-', color='#e53e3e', label='CV AUC (5-fold)', linewidth=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e53e3e')
ax.set_xlabel('Tamaño del conjunto de entrenamiento (submuestra 30%)')
ax.set_ylabel('AUC-ROC')
ax.set_title(
    f'Curva de aprendizaje — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}\n'
    f'Submuestra estratificada 30% del train',
    fontsize=11, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['curva_aprendizaje'] = figura_a_base64(fig)
plt.close(fig)

gap = train_mean[-1] - cv_mean[-1]
print(f'✅  ({time.time()-t0:.0f}s)')
print(f'Gap train-CV: {gap:.4f}  → '
      f'{"Posible overfitting" if gap > 0.05 else "Generalización correcta"}')

Submuestra: 8.068 filas (30% del train)  abandono=29.3%
✅  (42s)do curva de aprendizaje... 
Gap train-CV: 0.0435  → Generalización correcta


In [11]:
# ============================================================================
# CELDA 9: CALIBRACIÓN DE PROBABILIDADES + CURVAS ROC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_4 = [COLOR, '#e53e3e', '#38a169', '#d69e2e']

ax_cal = axes[0]
ax_cal.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Calibración perfecta')

ax_roc = axes[1]
ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1)

for idx, nombre_m in enumerate(['DecisionTree', 'RandomForest', 'ExtraTrees']):
    # Mejor estrategia de cada modelo según CV
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m    = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    # Calibración
    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-',
                color=colores_4[idx],
                label=f'{nombre_m} ({mejor_est})',
                linewidth=1.8, markersize=5)

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=colores_4[idx],
                label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=1.8)

ax_cal.set_xlabel('Probabilidad predicha')
ax_cal.set_ylabel('Fracción positivos reales')
ax_cal.set_title('Calibración de probabilidades', fontweight='bold')
ax_cal.legend(fontsize=8)
ax_cal.spines['top'].set_visible(False)
ax_cal.spines['right'].set_visible(False)

ax_roc.set_xlabel('Tasa de falsos positivos')
ax_roc.set_ylabel('Tasa de verdaderos positivos')
ax_roc.set_title('Curvas ROC comparativas', fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['calibracion_roc'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC generados')

✅ Calibración + ROC generados


In [12]:
# ============================================================================
# CELDA 10: MATRIZ DE CONFUSIÓN + CONVERGENCIA OPTUNA
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de confusión del mejor modelo
y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0, 1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0, 1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA} (test)',
                fontweight='bold')
ax_cm.set_ylabel('Etiqueta real')
ax_cm.set_xlabel('Etiqueta predicha')

# AUC por modelo (barras comparativas)
ax_bar = axes[1]
modelos_nombres = ['DecisionTree', 'RandomForest', 'ExtraTrees']
aucs_cv  = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in modelos_nombres]
colores_bar = [COLOR if m != MEJOR_MODELO else '#e53e3e' for m in modelos_nombres]
bars = ax_bar.bar(modelos_nombres, aucs_cv, color=colores_bar, edgecolor='white')
ax_bar.set_ylim(0.85, 0.92)
ax_bar.set_ylabel('AUC-ROC (mejor estrategia, CV)')
ax_bar.set_title('Comparativa AUC por modelo\n(rojo = mejor)', fontweight='bold')
for bar, val in zip(bars, aucs_cv):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Matriz de confusión + comparativa AUC generados')
print()
print('Classification report — mejor modelo en test:')
print(classification_report(y_test, y_pred_mejor,
      target_names=['Continúa', 'Abandona']))

✅ Matriz de confusión + comparativa AUC generados

Classification report — mejor modelo en test:
              precision    recall  f1-score   support

    Continúa       0.93      0.91      0.92      4758
    Abandona       0.79      0.84      0.82      1967

    accuracy                           0.89      6725
   macro avg       0.86      0.88      0.87      6725
weighted avg       0.89      0.89      0.89      6725



In [13]:
# ============================================================================
# CELDA 11: SERIALIZACIÓN DE RESULTADOS
# ============================================================================
# Separado del entrenamiento: si el HTML falla, los resultados están a salvo.
# ============================================================================

# Guardar parquet si no existe ya
if not (RUTA_RESULTS / 'results_arboles.parquet').exists():
    df_res.to_parquet(RUTA_RESULTS / 'results_arboles.parquet', index=False)
    print('✅ results_arboles.parquet guardado')
else:
    print('✅ results_arboles.parquet ya existía')

# Guardar excel si no existe ya    
df_res.to_excel(RUTA_RESULTS / 'results_arboles.xlsx', index=False)
print('✅ results_arboles.xlsx guardado')

# Guardar JSON con metadatos
meta = {
    'fecha'            : datetime.now().isoformat(),
    'familia'          : FAMILIA,
    'modelos'          : ['DecisionTree', 'RandomForest', 'ExtraTrees'],
    'estrategias'      : ESTRATEGIAS,
    'n_combinaciones'  : 9,
    'n_trials_optuna'  : N_TRIALS_OPTUNA,
    'cv_folds'         : N_SPLITS_CV,
    'random_state'     : RANDOM_STATE,
    'mejor_modelo'     : MEJOR_MODELO,
    'mejor_estrategia' : MEJOR_ESTRATEGIA,
    'mejor_auc_cv'     : round(float(df_cv.iloc[0]['auc_mean']), 4),
    'mejor_f1_cv'      : round(float(df_cv.iloc[0]['f1_mean']), 4),
    'mejores_params'   : PARAMS_OPTUNA,
    'auc_optuna'       : AUC_OPTUNA,
    'emisiones_co2_kg' : emisiones,
}
with open(RUTA_RESULTS / 'results_arboles.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

print('✅ results_arboles.json guardado')
print(f'✅ {len(modelos_fit)} modelos .pkl en {RUTA_MODELS}')

✅ results_arboles.parquet ya existía
✅ results_arboles.xlsx guardado
✅ results_arboles.json guardado
✅ 9 modelos .pkl en C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models


In [14]:
# ============================================================================
# CELDA 11b: AJUSTE DE UMBRAL ÓPTIMO
# ============================================================================
# Por defecto los clasificadores usan umbral=0.5 para decidir si un alumno
# abandona o no. Ese umbral es arbitrario y no considera el coste asimétrico:
#   - Falso negativo (predice no abandona cuando sí): peor consecuencia
#   - Falso positivo (predice abandona cuando no): menos grave
# Buscamos el umbral que maximiza F1 y el que garantiza Recall >= 0.75.
# ============================================================================

from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

RECALL_MINIMO    = 0.75
umbrales_result  = []
umbrales_rango   = np.arange(0.10, 0.91, 0.01)

print('=' * 60)
print('AJUSTE DE UMBRAL — búsqueda sobre test set')
print('=' * 60)

for nombre in ['DecisionTree', 'RandomForest', 'ExtraTrees']:
    mejor_est = (df_cv[df_cv['modelo'] == nombre]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    clave = nombre + '__' + mejor_est
    pipe  = modelos_fit[clave]

    try:
        y_proba = pipe.predict_proba(X_test_prep)[:, 1]
    except Exception:
        print(f'  ' + nombre + ': sin predict_proba — umbral omitido')
        continue

    # Umbral óptimo F1
    f1s     = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0)
               for t in umbrales_rango]
    idx_f1  = int(np.argmax(f1s))
    u_f1    = round(float(umbrales_rango[idx_f1]), 2)
    f1_opt  = round(float(f1s[idx_f1]), 4)

    # Umbral mínimo Recall >= RECALL_MINIMO
    recalls  = [recall_score(y_test, (y_proba >= t).astype(int), zero_division=0)
                for t in umbrales_rango]
    idx_rec  = next((j for j, r in enumerate(recalls) if r >= RECALL_MINIMO), None)
    u_rec    = round(float(umbrales_rango[idx_rec]), 2) if idx_rec is not None else None
    rec_rec  = round(float(recalls[idx_rec]), 4)        if idx_rec is not None else None
    pre_rec  = round(float(precision_score(
                   y_test, (y_proba >= umbrales_rango[idx_rec]).astype(int),
                   zero_division=0)), 4)                if idx_rec is not None else None

    # Métricas con umbral 0.5 (referencia)
    y05      = (y_proba >= 0.50).astype(int)
    f1_05    = round(float(f1_score(y_test, y05, zero_division=0)), 4)
    rec_05   = round(float(recall_score(y_test, y05, zero_division=0)), 4)
    ganancia = round(f1_opt - f1_05, 4)

    umbrales_result.append(dict(
        modelo         = nombre,
        estrategia     = mejor_est,
        umbral_f1      = u_f1,
        f1_umbral_opt  = f1_opt,
        f1_umbral_05   = f1_05,
        ganancia_f1    = ganancia,
        umbral_recall  = u_rec,
        recall_garantizado = rec_rec,
        precision_recall   = pre_rec,
        recall_umbral_05   = rec_05,
    ))

    print(f'  ' + nombre + ' (' + mejor_est + ')')
    print(f'    Umbral 0.50      → F1=' + str(f1_05) + '  Recall=' + str(rec_05))
    print(f'    Umbral ópt. F1  → ' + str(u_f1) + '  F1=' + str(f1_opt) +
          '  (+' + str(ganancia) + ')')
    if u_rec is not None:
        print(f'    Umbral Recall≥' + str(RECALL_MINIMO) + ' → ' + str(u_rec) +
              '  Recall=' + str(rec_rec) + '  Prec=' + str(pre_rec))
    print()

df_umbrales = pd.DataFrame(umbrales_result)
print('✅ Análisis de umbral completado')
print(df_umbrales[['modelo','umbral_f1','f1_umbral_opt','f1_umbral_05','ganancia_f1']].to_string(index=False))


AJUSTE DE UMBRAL — búsqueda sobre test set
  DecisionTree (smote)
    Umbral 0.50      → F1=0.7607  Recall=0.7966
    Umbral ópt. F1  → 0.57  F1=0.7618  (+0.0011)
    Umbral Recall≥0.75 → 0.1  Recall=0.9578  Prec=0.4484

  RandomForest (smote)
    Umbral 0.50      → F1=0.8174  Recall=0.8419
    Umbral ópt. F1  → 0.48  F1=0.8181  (+0.0007)
    Umbral Recall≥0.75 → 0.1  Recall=0.9842  Prec=0.4594

  ExtraTrees (none)
    Umbral 0.50      → F1=0.761  Recall=0.6797
    Umbral ópt. F1  → 0.38  F1=0.7882  (+0.0272)
    Umbral Recall≥0.75 → 0.1  Recall=0.9817  Prec=0.4427

✅ Análisis de umbral completado
      modelo  umbral_f1  f1_umbral_opt  f1_umbral_05  ganancia_f1
DecisionTree       0.57         0.7618        0.7607       0.0011
RandomForest       0.48         0.8181        0.8174       0.0007
  ExtraTrees       0.38         0.7882        0.7610       0.0272


In [15]:
# ============================================================================
# CELDA 12: GENERAR HTML
# ============================================================================
# Todo el texto se genera automaticamente desde los resultados en memoria.
# Si cambian los datos y resultados, cambia el texto automaticamente.
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm02_arboles.html'

# --- KPIs ---
kpis = [
    {'valor': '3',                                 'titulo': 'Modelos'},
    {'valor': '3',                                 'titulo': 'Estrategias'},
    {'valor': '9',                                'titulo': 'Combinaciones'},
    {'valor': str(N_TRIALS_OPTUNA),                'titulo': 'Trials Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}',  'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',   'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# --- Texto dinamico: calculado desde df_cv ---
segundo     = df_cv.iloc[1]
diff_auc    = df_cv.iloc[0]['auc_mean'] - segundo['auc_mean']
diff_f1     = df_cv.iloc[0]['f1_mean']  - segundo['f1_mean']
peor        = df_cv.iloc[-1]
rango_auc   = df_cv.iloc[0]['auc_mean'] - peor['auc_mean']
sub_mejor   = df_cv[df_cv['modelo'] == MEJOR_MODELO]
auc_none    = sub_mejor[sub_mejor['estrategia'] == 'none']['auc_mean'].values[0]
mejora_bal  = df_cv.iloc[0]['auc_mean'] - auc_none

# Texto umbral
if 'df_umbrales' in dir() and not df_umbrales.empty:
    mejor_u = df_umbrales.loc[df_umbrales['ganancia_f1'].idxmax()]
    parrafo_umbral = (
        '<p><strong>Ajuste de umbral:</strong> el umbral por defecto (0.50) puede no ser óptimo '
        'dado el coste asimétrico de los errores. El umbral que maximiza F1 para '
        '<strong>' + str(mejor_u['modelo']) + '</strong> es '
        '<strong>' + str(mejor_u['umbral_f1']) + '</strong>, '
        'mejorando el F1 en <strong>+' + str(mejor_u['ganancia_f1']) + '</strong> puntos. '
        'Para garantizar un Recall &ge; 0.75 (detectar al menos 3 de cada 4 abandonos), '
        'el umbral recomendado es <strong>' + str(mejor_u['umbral_recall']) + '</strong>.</p>'
    )
else:
    parrafo_umbral = ''

texto_dinamico = (
    f'<p>El analisis de <strong>{len(df_cv)} combinaciones</strong> '
    f'(3 modelos x 3 estrategias de balance) revela que '
    f'<strong>{MEJOR_MODELO}</strong> con estrategia <strong>{MEJOR_ESTRATEGIA}</strong> '
    f'es la combinacion ganadora de la familia lineal, alcanzando un '
    f'<strong>AUC CV de {df_cv.iloc[0]["auc_mean"]:.4f} '
    f'&plusmn; {df_cv.iloc[0]["auc_std"]:.4f}</strong> '
    f'y un <strong>F1 de {df_cv.iloc[0]["f1_mean"]:.4f}</strong>.</p>'
    f'<p>Supera al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) en '
    f'<strong>{diff_auc:.4f} puntos de AUC</strong> y <strong>{diff_f1:.4f} de F1</strong>. '
    f'El rango total entre mejor y peor combinacion es de {rango_auc:.4f} puntos de AUC, '
    f'lo que indica que la eleccion del modelo tiene mas impacto que la estrategia de balance.</p>'
    f'<p>El ajuste de balance mejora {MEJOR_MODELO} en <strong>{mejora_bal:.4f} puntos de AUC</strong> '
    f'respecto a entrenar sin ajuste, confirmando que el desbalance 70/30 del dataset '
    f'penaliza a los modelos lineales cuando no se corrige.</p>'
)

# --- Tabla pivotada: 1 fila por modelo, mejor estrategia ---
modelos_orden = ['DecisionTree', 'RandomForest', 'ExtraTrees']
filas_pivot = ''
for modelo in modelos_orden:
    sub      = df_cv[df_cv['modelo'] == modelo].iloc[0]
    es_mejor = modelo == MEJOR_MODELO
    bg       = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    estrella = ' 🏆' if es_mejor else ''
    auc_opt_str = f'{AUC_OPTUNA[modelo]:.4f}' if AUC_OPTUNA[modelo] is not None else '—'
    filas_pivot += (
        f'<tr style="{bg}">'
        f'<td>{modelo}{estrella}</td>'
        f'<td>{sub["estrategia"]}</td>'
        f'<td>{sub["auc_mean"]:.4f} &plusmn; {sub["auc_std"]:.4f}</td>'
        f'<td>{sub["f1_mean"]:.4f}</td>'
        f'<td>{sub["precision_mean"]:.4f}</td>'
        f'<td>{sub["recall_mean"]:.4f}</td>'
        f'<td>{auc_opt_str}</td>'
        f'<td><code>{json.dumps(PARAMS_OPTUNA[modelo])}</code></td>'
        f'</tr>'
    )
tabla_pivot = (
    '<p>Una fila por modelo con su mejor estrategia. '
    '🏆 = mejor combinacion global.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean&plusmn;std)</th><th>F1 CV</th>'
    '<th>Precision</th><th>Recall</th>'
    '<th>AUC Optuna</th><th>Hiperparametros</th>'
    f'</tr></thead><tbody>{filas_pivot}</tbody></table>'
)

# --- Tabla completa CV (12 filas) ---
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}">'
        f'<td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    f'<p>Ordenado por AUC-ROC descendente. Fila destacada = mejor combinacion: '
    f'<strong>{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}</strong> '
    f'(AUC={df_cv.iloc[0]["auc_mean"]:.4f}).</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean&plusmn;std)</th>'
    '<th>F1</th><th>Precision</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

# --- Hiperparametros Optuna ---
filas_params_list = []
for m in ['DecisionTree', 'RandomForest', 'ExtraTrees']:
    auc_str = f'{AUC_OPTUNA[m]:.4f}' if AUC_OPTUNA[m] is not None else '—'
    filas_params_list.append(
        f'<tr><td>{m}</td><td><code>{json.dumps(PARAMS_OPTUNA[m])}</code></td>'
        f'<td>{auc_str}</td></tr>'
    )
filas_params = ''.join(filas_params_list)
tabla_params = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Hiperparametros optimos (Optuna)</th><th>AUC busqueda</th>'
    f'</tr></thead><tbody>{filas_params}</tbody></table>'
)

# --- Coeficientes y CO2 ---
coef_html = (
    f'<img src="data:image/png;base64,{graficos_b64.get("importancia","")}" style="max-width:100%">'
    if 'importancia' in graficos_b64
    else '<p>ℹ️ El mejor modelo no expone importancia de features.</p>'
)
co2_html = (
    f'<p>🌿 Huella CO2 del entrenamiento: <strong>{emisiones:.6f} kg CO2eq</strong></p>'
    if emisiones
    else '<p>ℹ️ Modelos cargados desde disco - sin nuevo entrenamiento en esta ejecucion.</p>'
)


def img_tag(key, caption=''):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    cap = f'<p style="font-size:0.85rem;color:#666;margin-top:4px;">{caption}</p>' if caption else ''
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">{cap}'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + kpis_html + '</section>'
    '<section class="seccion"><h2>Conclusiones del analisis</h2>' + texto_dinamico + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo — mejor estrategia de cada uno</h2>' + tabla_pivot + '</section>'
    '<section class="seccion"><h2>Resultados completos 5-Fold CV — 9 combinaciones</h2>' + tabla_cv + '</section>'
    f'<section class="seccion"><h2>Hiperparametros optimos — Optuna ({N_TRIALS_OPTUNA} trials x 4 modelos)</h2>' + tabla_params + '</section>'
    '<section class="seccion"><h2>Importancia de features — Top 10 features</h2>'
    '<p>Exclusivo de la familia lineal. Rojo = feature más importante. Gini importance del mejor modelo.</p>'
    + coef_html + '</section>'
    f'<section class="seccion"><h2>Curva de aprendizaje — {MEJOR_MODELO}</h2>'
    '<p>Submuestra estratificada 30% del train. Convergencia entre train y CV indica buena generalizacion.</p>'
    + img_tag("curva_aprendizaje") + '</section>'
    '<section class="seccion"><h2>Calibracion de probabilidades y curvas ROC</h2>'
    + img_tag("calibracion_roc") + '</section>'
    '<section class="seccion"><h2>Matriz de confusion y comparativa AUC</h2>'
    + img_tag("cm_comparativa") + '</section>'
    '<section class="seccion"><h2>Sostenibilidad computacional</h2>' + co2_html + '</section>'
)

render_pagina_desde_fichero(
    'f5_m02_arboles.ipynb',
    secciones
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m02_arboles.html


In [16]:
# ============================================================================
# CELDA 13: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m02_arboles')
print('=' * 60)
print(f'Familia     : Árboles de decisión')
print(f'Modelos     : DecisionTree · RandomForest · ExtraTrees')
print(f'Estrategias : none · balanced · smote  (9 combinaciones)')
print(f'Optuna      : {N_TRIALS_OPTUNA} trials registrados  |  AUC métrica')
print()
print(f'🏆 Mejor : {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('📁 Archivos:')
print('   data/05_modelado/results/results_arboles.parquet')
print('   data/05_modelado/results/results_arboles.xlsx')
print('   data/05_modelado/results/results_arboles.json')
print('   data/05_modelado/models/  (12 × .pkl)')
print('   docs/html/fase5/m02_arboles.html')
print()
print('➡️  Siguiente: f5_m03_boosting.ipynb')

RESUMEN FINAL — f5_m02_arboles
Familia     : Árboles de decisión
Modelos     : DecisionTree · RandomForest · ExtraTrees
Estrategias : none · balanced · smote  (9 combinaciones)
Optuna      : 50 trials registrados  |  AUC métrica

🏆 Mejor : RandomForest + smote
   AUC CV = 0.9473 ± 0.0028
   F1  CV = 0.8170 ± 0.0040

📁 Archivos:
   data/05_modelado/results/results_arboles.parquet
   data/05_modelado/results/results_arboles.xlsx
   data/05_modelado/results/results_arboles.json
   data/05_modelado/models/  (12 × .pkl)
   docs/html/fase5/m02_arboles.html

➡️  Siguiente: f5_m03_boosting.ipynb
